# Spatial Density Analysis (KDE)

This notebook generates smooth heatmaps of your location history using **Kernel Density Estimation (KDE)**.
It highlights areas of high activity without adhering strictly to the road network.


In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KernelDensity
import geopandas as gpd
from shapely.geometry import Point
from shapely.prepared import prep
import osmnx as ox
from dateutil import parser as dtparser

os.makedirs('outputs', exist_ok=True)


In [ ]:
# --- Config ---
CITY_NAME = 'Beirut, Lebanon'
CELL_SIZE_M = 100   # Resolution (meters)
BANDWIDTH_M = 300   # KDE smoothing radius

# Load Data
with open('location-history.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Basic Geo Parser
def parse_geo(s):
    if not s: return None
    s = s.replace('geo:', '')
    try: return [float(x) for x in s.split(',')]
    except: return None

pts = []
for ep in data:
    if 'timelinePath' in ep:
        for p in ep['timelinePath']:
            geo = parse_geo(p.get('point'))
            if geo:
                pts.append(geo)

df = pd.DataFrame(pts, columns=['lat', 'lon'])
print(f'Loaded {len(df)} points for KDE Analysis.')


## 1. Setup Grid
Creating a meshgrid over the city area to evaluate density.


In [ ]:
# Get city bounds in Metric CRS
city_gdf = ox.geocode_to_gdf(CITY_NAME)
utm_crs = city_gdf.estimate_utm_crs()
city_m = city_gdf.to_crs(utm_crs)
poly_m = city_m.geometry.iloc[0]

minx, miny, maxx, maxy = poly_m.bounds
xs = np.arange(minx, maxx, CELL_SIZE_M)
ys = np.arange(miny, maxy, CELL_SIZE_M)
xx, yy = np.meshgrid(xs, ys)
grid_points = np.vstack([xx.ravel(), yy.ravel()]).T

# Poly mask to clip sea/outside areas
prep_poly = prep(poly_m)
mask = np.array([prep_poly.contains(Point(p)) for p in grid_points]).reshape(xx.shape)
print(f'Grid shape: {xx.shape}')


In [ ]:
# Convert points to Metric CRS
gdf_pts = gpd.GeoDataFrame(df, geometry=[Point(xy) for xy in zip(df.lon, df.lat)], crs='EPSG:4326')
gdf_pts = gdf_pts.to_crs(utm_crs)
gdf_pts = gdf_pts[gdf_pts.within(poly_m)] # Filter to city
print(f'Points within city: {len(gdf_pts)}')


## 2. Generate Heatmap
Calculating density using Gaussian Kernels.


In [ ]:
if not gdf_pts.empty:
    coords = np.vstack([gdf_pts.geometry.x, gdf_pts.geometry.y]).T
    
    print('Fitting KDE... (this may take a moment)')
    kde = KernelDensity(bandwidth=BANDWIDTH_M, kernel='gaussian')
    kde.fit(coords)
    
    print('Evaluating scoring...')
    # Score samples returns log-density
    log_dens = kde.score_samples(grid_points)
    dens = np.exp(log_dens).reshape(xx.shape)
    
    # Apply Mask
    dens_masked = np.where(mask, dens, np.nan)
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.imshow(dens_masked, origin='lower', cmap='inferno', extent=[minx, maxx, miny, maxy])
    plt.title(f'Mobility Density Heatmap (Bandwidth={BANDWIDTH_M}m)')
    plt.colorbar(label='Density')
    plt.axis('off')
    plt.savefig('outputs/kde_heatmap.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved outputs/kde_heatmap.png')
else:
    print('No points found in city boundary.')


## 3. Hexbin Analysis
Discrete binning for high-contrast density visualization.


In [ ]:
if not gdf_pts.empty:
    fig, ax = plt.subplots(figsize=(12, 10))
    ax.set_axis_off()
    ax.set_title('Hexbin Density Map', fontsize=16)
    
    # Extract x/y
    x = gdf_pts.geometry.x
    y = gdf_pts.geometry.y
    
    hb = ax.hexbin(x, y, gridsize=50, cmap='magma', mincnt=1, bins='log')
    plt.colorbar(hb, ax=ax, label='Log Points per Bin')
    
    plt.savefig('outputs/hexbin_map.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved outputs/hexbin_map.png')
